# Day 5: Introduction to Machine Learning & Capstone Project

**Python for Data Science Workshop — Day 5 of 5**  
**Instructor:** Dr. Yaé Ulrich Gaba  
**Duration:** ~4 hours  

---

### Workshop Recap

| Day | Topic | Status |
|-----|-------|--------|
| Day 1 | Python Fundamentals | ✓ Done |
| Day 2 | Data Structures & NumPy | ✓ Done |
| Day 3 | Pandas & Data Wrangling | ✓ Done |
| Day 4 | Data Visualization | ✓ Done |
| **Day 5** | **Machine Learning & Capstone** | ← Today |

---

### Today's Agenda

1. Introduction to scikit-learn & the ML workflow  
2. Linear Regression — predicting life expectancy  
3. Classification — logistic regression & decision trees  
4. Model Evaluation — metrics, confusion matrix, cross-validation  
5. Capstone Mini-Project — end-to-end African development pipeline  
6. Workshop Summary & Next Steps  

---

### Learning Objectives

By the end of this session, you will be able to:

1. Describe the supervised ML workflow: data → split → train → evaluate → improve
2. Build and interpret a **Linear Regression** model with scikit-learn
3. Build **Logistic Regression** and **Decision Tree** classifiers
4. Evaluate models using accuracy, precision, recall, F1-score, confusion matrix, and cross-validation
5. Complete a full end-to-end ML pipeline on an African development dataset

---

> **Why Africa-focused data?**  
> Data Science is most powerful when applied to problems that matter to your community.  
> Today we use realistic data inspired by African countries to practice techniques that are directly relevant to research, policy, and development work on the continent.

In [ ]:
# Standard imports — run this first!
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid', font_scale=1.05)
np.random.seed(42)

print("Libraries loaded successfully!")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")

---

## Part 1: Introduction to scikit-learn & the ML Workflow

### 1.1 What is Machine Learning?

Machine Learning is the practice of writing programs that **learn patterns from data** rather than following hand-written rules.

Instead of:
```
Rule: IF GDP_per_capita > 5000 AND health_spending > 200 THEN life_expectancy > 70
```

We do:
```
model.fit(X_data, y_data)   # let the algorithm discover the rule!
```

### 1.2 Three Types of Machine Learning

| Type | What it does | Example |
|------|-------------|--------|
| **Supervised** | Learns from labeled examples | Predict crop yield from rainfall |
| **Unsupervised** | Finds structure without labels | Group countries by development patterns |
| **Reinforcement** | Learns through trial and reward | Game-playing AI |

Today we focus on **supervised learning**.

### 1.3 The Standard ML Workflow

```
  Raw Data
     │
     ▼
  Clean & Explore (EDA)
     │
     ▼
  Feature Engineering
     │
     ▼
  Train / Test Split
     │
     ├──► Training Set ──► model.fit(X_train, y_train)
     │
     └──► Test Set ──────► model.predict(X_test) ──► Evaluate
                                                         │
                                               Iterate & Improve
```

### 1.4 Why Do We Split Into Train and Test?

If you evaluate a model on the **same data it was trained on**, you cannot tell whether it has truly learned  
generalizable patterns or simply **memorized** the training data.

- **Training set** (~80%): The model learns from this.
- **Test set** (~20%): Unseen data used only for final evaluation.

**Rule of thumb:** Never "peek" at the test set during model development.

In [ ]:
# Import the main scikit-learn tools we will use today
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score
)
import warnings
warnings.filterwarnings('ignore')

print("All scikit-learn modules loaded!")

### 1.5 The scikit-learn API: fit / predict / score

Every scikit-learn model follows the **same three-step API**. Once you learn it for one model, you know it for all:

```python
from sklearn.some_module import SomeModel

model = SomeModel(hyperparameters)   # 1. Create
model.fit(X_train, y_train)          # 2. Train
predictions = model.predict(X_test)  # 3. Predict
score = model.score(X_test, y_test)  # 4. Evaluate
```

This consistency is one of scikit-learn's greatest strengths.

---

## Part 2: Linear Regression — Predicting Life Expectancy

### 2.1 What is Linear Regression?

Linear regression predicts a **continuous** numerical target by fitting a straight line (or hyperplane) through the data.

The formula for simple linear regression:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_n x_n$$

Where:
- $\hat{y}$ is the predicted value (e.g., life expectancy in years)
- $\beta_0$ is the intercept
- $\beta_1, \ldots, \beta_n$ are the coefficients learned from data
- $x_1, \ldots, x_n$ are the input features (e.g., GDP per capita, health spending)

### 2.2 Creating Our African Country Dataset

We create a **synthetic but realistic** dataset of 54 African countries with features known to influence life expectancy.  
The values are based on ranges typical of sub-Saharan and North African countries.

In [ ]:
# Create a synthetic African country health & development dataset
np.random.seed(42)
n_countries = 54  # 54 African Union member states

african_countries = [
    "Algeria", "Angola", "Benin", "Botswana", "Burkina Faso",
    "Burundi", "Cabo Verde", "Cameroon", "CAR", "Chad",
    "Comoros", "DRC", "Djibouti", "Egypt", "Equatorial Guinea",
    "Eritrea", "Eswatini", "Ethiopia", "Gabon", "Gambia",
    "Ghana", "Guinea", "Guinea-Bissau", "Ivory Coast", "Kenya",
    "Lesotho", "Liberia", "Libya", "Madagascar", "Malawi",
    "Mali", "Mauritania", "Mauritius", "Morocco", "Mozambique",
    "Namibia", "Niger", "Nigeria", "Rwanda", "Sao Tome",
    "Senegal", "Seychelles", "Sierra Leone", "Somalia", "South Africa",
    "South Sudan", "Sudan", "Tanzania", "Togo", "Tunisia",
    "Uganda", "Zambia", "Zimbabwe", "Eritrea-2"
]

# Features drawn from realistic distributions for African countries
gdp_per_capita  = np.random.lognormal(mean=7.5, sigma=0.9, size=n_countries)   # USD
health_spending = np.random.uniform(20, 350, n_countries)   # USD per capita per year
literacy_rate   = np.clip(np.random.normal(65, 20, n_countries), 20, 98)        # %
physicians      = np.clip(np.random.exponential(0.5, n_countries), 0.02, 3.5)   # per 1000 people
urban_pct       = np.clip(np.random.normal(45, 18, n_countries), 12, 90)        # % urban
sanitation_pct  = np.clip(np.random.normal(50, 25, n_countries), 8, 97)         # % with access
safe_water_pct  = np.clip(np.random.normal(70, 18, n_countries), 30, 99)        # % with access

# Life expectancy: realistic linear combination + noise
life_expectancy = (
    40
    + 3.5  * np.log(gdp_per_capita)
    + 0.04 * health_spending
    + 0.10 * literacy_rate
    + 2.5  * physicians
    + 0.05 * sanitation_pct
    + 0.04 * safe_water_pct
    + np.random.normal(0, 2.5, n_countries)
)
life_expectancy = np.clip(life_expectancy, 48, 80)

africa_df = pd.DataFrame({
    'country':          african_countries,
    'gdp_per_capita':   np.round(gdp_per_capita, 0),
    'health_spending':  np.round(health_spending, 1),
    'literacy_rate':    np.round(literacy_rate, 1),
    'physicians_per1k': np.round(physicians, 3),
    'urban_pct':        np.round(urban_pct, 1),
    'sanitation_pct':   np.round(sanitation_pct, 1),
    'safe_water_pct':   np.round(safe_water_pct, 1),
    'life_expectancy':  np.round(life_expectancy, 1)
})

print(f"Dataset shape: {africa_df.shape}")
print(f"\nLife expectancy range: {africa_df.life_expectancy.min():.1f} — {africa_df.life_expectancy.max():.1f} years")
print(f"Mean life expectancy: {africa_df.life_expectancy.mean():.1f} years")
africa_df.head(10)

In [ ]:
# Quick statistical summary
africa_df.describe().round(2)

In [ ]:
# Visualise how individual features relate to life expectancy
features_to_plot = ['gdp_per_capita', 'health_spending', 'literacy_rate',
                    'physicians_per1k', 'sanitation_pct', 'safe_water_pct']
titles = ['GDP per Capita (USD)', 'Health Spending (USD/yr)',
          'Literacy Rate (%)', 'Physicians per 1,000',
          'Sanitation Access (%)', 'Safe Water Access (%)']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, feat, title in zip(axes.flatten(), features_to_plot, titles):
    ax.scatter(africa_df[feat], africa_df['life_expectancy'],
               color='#1a6e9e', alpha=0.7, edgecolors='white', linewidth=0.5, s=60)
    # Add trend line
    z = np.polyfit(africa_df[feat], africa_df['life_expectancy'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(africa_df[feat].min(), africa_df[feat].max(), 100)
    ax.plot(x_line, p(x_line), 'r--', linewidth=2, alpha=0.8)
    ax.set_xlabel(title, fontsize=10)
    ax.set_ylabel('Life Expectancy (years)', fontsize=10)
    corr = africa_df[feat].corr(africa_df['life_expectancy'])
    ax.set_title(f'r = {corr:.3f}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

fig.suptitle('Feature Correlations with Life Expectancy (African Countries)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Prepare features and target for regression
feature_cols = ['gdp_per_capita', 'health_spending', 'literacy_rate',
                'physicians_per1k', 'sanitation_pct', 'safe_water_pct']

X_life = africa_df[feature_cols].copy()
y_life = africa_df['life_expectancy'].copy()

# Train / test split  (80% train, 20% test)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_life, y_life, test_size=0.20, random_state=42
)

print(f"Training samples : {X_train_l.shape[0]}")
print(f"Test samples     : {X_test_l.shape[0]}")
print(f"Features         : {list(feature_cols)}")

In [ ]:
# Train Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_l, y_train_l)

# Predict
y_pred_l = lin_reg.predict(X_test_l)

# Metrics
mse  = mean_squared_error(y_test_l, y_pred_l)
rmse = np.sqrt(mse)
r2   = r2_score(y_test_l, y_pred_l)

print("=== Linear Regression Results ===")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% of variance explained)")
print(f"RMSE : {rmse:.2f} years  (average prediction error)")
print(f"MSE  : {mse:.4f}")
print()

# Coefficients
coef_df = pd.DataFrame({
    'Feature':     feature_cols,
    'Coefficient': lin_reg.coef_
}).sort_values('Coefficient', ascending=False)

print("Coefficients (impact per unit increase):")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {lin_reg.intercept_:.2f}")

In [ ]:
# Visualise regression results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1 — Actual vs Predicted
axes[0].scatter(y_test_l, y_pred_l, color='#1a6e9e', s=80,
               edgecolors='white', linewidth=0.8, alpha=0.85, zorder=5)
lims = [min(y_test_l.min(), y_pred_l.min()) - 1,
        max(y_test_l.max(), y_pred_l.max()) + 1]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Life Expectancy (years)', fontsize=11)
axes[0].set_ylabel('Predicted Life Expectancy (years)', fontsize=11)
axes[0].set_title(f'Actual vs Predicted (R²={r2:.3f})', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2 — Residuals vs Predicted
residuals = y_test_l.values - y_pred_l
axes[1].scatter(y_pred_l, residuals, color='#e07b39', s=80,
               edgecolors='white', linewidth=0.8, alpha=0.85)
axes[1].axhline(0, color='black', linewidth=1.5, linestyle='-')
axes[1].set_xlabel('Predicted Life Expectancy (years)', fontsize=11)
axes[1].set_ylabel('Residual (years)', fontsize=11)
axes[1].set_title(f'Residual Plot (RMSE = {rmse:.2f} yrs)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Plot 3 — Coefficient bar chart
colors_coef = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
axes[2].barh(coef_df['Feature'], coef_df['Coefficient'],
             color=colors_coef, edgecolor='white', height=0.6)
axes[2].axvline(0, color='black', linewidth=0.8)
axes[2].set_xlabel('Coefficient Value', fontsize=11)
axes[2].set_title('Feature Coefficients', fontweight='bold')
axes[2].grid(axis='x', alpha=0.3)

fig.suptitle('Linear Regression — Life Expectancy Model', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Interpreting the Coefficients

Each coefficient tells us: **holding all other features constant, how much does life expectancy change when this feature increases by 1 unit?**

For example:
- A coefficient of **+0.10** for `literacy_rate` means: 1% increase in literacy rate → +0.10 years of life expectancy (on average)
- Features with **positive** coefficients improve life expectancy
- Features with **negative** coefficients reduce it (if any appear)

> **Note on scale:** Raw coefficients depend on the scale of each feature. A very small coefficient for `gdp_per_capita` does not mean it is unimportant — GDP varies by thousands of dollars. To compare importance fairly, you would need to **standardize** the features first.

---

## Part 3: Classification — Predicting Development Level

### 3.1 Regression vs Classification

| Task | Target Type | Example |
|------|------------|--------|
| **Regression** | Continuous number | Life expectancy (e.g., 63.4 years) |
| **Classification** | Category / class | Development level (Low / High) |

### 3.2 Creating the Classification Dataset

We create a richer dataset of **200 synthetic African data points** and classify whether a country/region  
falls into a **High Development** or **Low Development** category based on a Human Development Index (HDI) proxy.

In [ ]:
# Create classification dataset — 200 data points inspired by African country indicators
np.random.seed(7)
n = 200

# Draw features from realistic African ranges
gdp          = np.random.lognormal(mean=7.4, sigma=0.85, size=n)
edu_index    = np.clip(np.random.normal(0.50, 0.15, n), 0.10, 0.90)   # 0–1 scale
health_idx   = np.clip(np.random.normal(0.55, 0.12, n), 0.20, 0.90)   # 0–1 scale
internet_pct = np.clip(np.random.normal(35, 22, n), 1, 90)             # % population
gini         = np.clip(np.random.normal(43, 9, n), 25, 65)             # Gini coefficient
child_mort   = np.clip(np.random.exponential(40, n), 5, 150)           # per 1000 live births
school_years = np.clip(np.random.normal(7.5, 2.5, n), 2, 14)           # mean years of schooling

# HDI proxy: composite score used to determine class label
hdi_score = (
    0.25 * np.log(gdp) / np.log(gdp.max())
    + 0.25 * edu_index
    + 0.25 * health_idx
    + 0.15 * internet_pct / 100
    - 0.05 * gini / 65
    - 0.05 * child_mort / 150
    + np.random.normal(0, 0.04, n)
)

# Binary label: 1 = High Development, 0 = Low Development
label = (hdi_score > np.median(hdi_score)).astype(int)

class_df = pd.DataFrame({
    'gdp_per_capita':  np.round(gdp, 0),
    'edu_index':       np.round(edu_index, 3),
    'health_index':    np.round(health_idx, 3),
    'internet_pct':    np.round(internet_pct, 1),
    'gini_coeff':      np.round(gini, 1),
    'child_mortality': np.round(child_mort, 1),
    'school_years':    np.round(school_years, 1),
    'high_development': label
})

print(f"Dataset: {class_df.shape[0]} samples, {class_df.shape[1]} columns")
print(f"\nClass distribution:")
print(class_df['high_development'].value_counts().rename({0: 'Low Development', 1: 'High Development'}))
print(f"\nClass balance: {label.mean()*100:.1f}% High Development")
class_df.head(8)

In [ ]:
# Exploratory plot — key features by class
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
feat_names = ['gdp_per_capita', 'edu_index', 'health_index',
              'internet_pct', 'child_mortality', 'school_years']
feat_labels = ['GDP per Capita (USD)', 'Education Index',
               'Health Index', 'Internet Access (%)',
               'Child Mortality (per 1000)', 'Mean School Years']
palette = {0: '#e74c3c', 1: '#2ecc71'}

for ax, feat, label in zip(axes.flatten(), feat_names, feat_labels):
    sns.boxplot(data=class_df, x='high_development', y=feat,
                palette=palette, ax=ax, width=0.5)
    ax.set_xticklabels(['Low Dev.', 'High Dev.'])
    ax.set_xlabel('')
    ax.set_ylabel(label, fontsize=10)
    ax.set_title(label, fontweight='bold', fontsize=11)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Feature Distributions by Development Class',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Prepare features and target
X_cls = class_df.drop(columns=['high_development'])
y_cls = class_df['high_development']

# Train / test split — stratified to preserve class balance
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.20, random_state=42, stratify=y_cls
)

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_c)    # fit only on training data!
X_test_scaled  = scaler.transform(X_test_c)          # apply same scale to test

print(f"Training : {X_train_c.shape[0]} samples")
print(f"Test     : {X_test_c.shape[0]} samples")
print(f"\nTraining class distribution:")
print(y_train_c.value_counts().rename({0: 'Low', 1: 'High'}))

### 3.3 Logistic Regression

Despite its name, **Logistic Regression is a classification** algorithm. It models the **probability** that a sample belongs to a class:

$$P(y=1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \ldots + \beta_n x_n)}}$$

The **sigmoid function** squeezes the output into (0, 1), which we interpret as a probability.  
If $P(y=1) > 0.5$, we predict class 1 (High Development); otherwise class 0 (Low Development).

> Logistic regression requires **feature scaling** because it uses gradient-based optimization.

In [ ]:
# Train Logistic Regression
log_reg = LogisticRegression(max_iter=500, random_state=42, C=1.0)
log_reg.fit(X_train_scaled, y_train_c)

y_pred_lr = log_reg.predict(X_test_scaled)
acc_lr    = accuracy_score(y_test_c, y_pred_lr)

print(f"Logistic Regression Accuracy: {acc_lr:.4f}  ({acc_lr*100:.1f}%)")

# Show predicted probabilities for first 10 test samples
proba = log_reg.predict_proba(X_test_scaled)[:10]
print("\nPredicted Probabilities (first 10 test samples):")
print(f"{'P(Low Dev)':>12}  {'P(High Dev)':>12}  {'Prediction':>12}  {'Actual':>8}")
print("-" * 52)
for p, pred, actual in zip(proba, y_pred_lr[:10], y_test_c.values[:10]):
    pred_label = 'High' if pred == 1 else 'Low'
    actual_label = 'High' if actual == 1 else 'Low'
    correct = '✓' if pred == actual else '✗'
    print(f"{p[0]:>12.4f}  {p[1]:>12.4f}  {pred_label:>12}  {actual_label:>8}  {correct}")

### 3.4 Decision Tree Classifier

A decision tree makes predictions by learning a series of **if/else rules** from the data — much like a flowchart.

**Advantages:**
- Highly **interpretable** — you can draw and read the tree
- Handles non-linear relationships naturally
- Does **not** require feature scaling

**Watch out for:**
- **Overfitting** if the tree grows too deep (memorizes training data)
- We control this with `max_depth`

In [ ]:
# Train Decision Tree — no scaling needed!
dt_clf = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
dt_clf.fit(X_train_c, y_train_c)

y_pred_dt = dt_clf.predict(X_test_c)
acc_dt    = accuracy_score(y_test_c, y_pred_dt)

print(f"Decision Tree Accuracy: {acc_dt:.4f}  ({acc_dt*100:.1f}%)")

# Feature importance
importance_dt = pd.Series(
    dt_clf.feature_importances_,
    index=X_cls.columns
).sort_values(ascending=False)

print("\nFeature Importances (Decision Tree):")
for feat, imp in importance_dt.items():
    bar = '█' * int(imp * 40)
    print(f"  {feat:<20} {imp:.4f}  {bar}")

In [ ]:
# Visualise the decision tree (first 3 levels for readability)
fig, ax = plt.subplots(figsize=(20, 8))

plot_tree(
    dt_clf,
    feature_names=list(X_cls.columns),
    class_names=['Low Dev.', 'High Dev.'],
    filled=True,
    max_depth=3,       # show only top 3 levels
    fontsize=9,
    impurity=False,
    proportion=False,
    ax=ax
)

ax.set_title(
    'Decision Tree (first 3 levels shown)\n'
    'Green = High Development  |  Orange = Low Development',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

---

## Part 4: Model Evaluation

### 4.1 Why Accuracy Alone Is Not Enough

Imagine a disease affects only 5% of patients. A model that **always predicts "no disease"** achieves **95% accuracy** — yet it is completely useless.

We need more nuanced metrics:

| Metric | Formula | Meaning |
|--------|---------|--------|
| **Accuracy** | (TP + TN) / Total | Overall correct predictions |
| **Precision** | TP / (TP + FP) | Of all "Positive" predictions, how many were right? |
| **Recall** | TP / (TP + FN) | Of all actual Positives, how many did we find? |
| **F1-Score** | 2 × (P × R) / (P + R) | Harmonic mean of precision and recall |

Where:
- **TP** = True Positives (correctly predicted positive)
- **TN** = True Negatives (correctly predicted negative)
- **FP** = False Positives (predicted positive but actually negative — Type I error)
- **FN** = False Negatives (predicted negative but actually positive — Type II error)

### 4.2 Confusion Matrix

In [ ]:
# Confusion matrices for both classifiers side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

class_names = ['Low Dev.', 'High Dev.']
models_preds = [
    ('Logistic Regression', y_pred_lr, acc_lr),
    ('Decision Tree (depth=4)', y_pred_dt, acc_dt)
]

for ax, (name, y_pred, acc) in zip(axes, models_preds):
    cm = confusion_matrix(y_test_c, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAccuracy: {acc*100:.1f}%', fontweight='bold', fontsize=12)

fig.suptitle('Confusion Matrices — Development Classification',
             fontsize=14, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

print("\nReading a confusion matrix:")
print("  Top-left  (TN): Correctly predicted Low Development")
print("  Top-right (FP): Predicted High — actually Low (False Alarm)")
print("  Bot-left  (FN): Predicted Low  — actually High (Missed)")
print("  Bot-right (TP): Correctly predicted High Development")

In [ ]:
# Detailed classification reports
for name, y_pred in [('Logistic Regression', y_pred_lr), ('Decision Tree', y_pred_dt)]:
    print(f"{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(classification_report(y_test_c, y_pred, target_names=class_names))

# Summary table of key metrics
metrics_summary = []
for name, y_pred in [('Logistic Regression', y_pred_lr), ('Decision Tree', y_pred_dt)]:
    metrics_summary.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_test_c, y_pred),
        'Precision': precision_score(y_test_c, y_pred),
        'Recall':    recall_score(y_test_c, y_pred),
        'F1-Score':  f1_score(y_test_c, y_pred)
    })

metrics_df = pd.DataFrame(metrics_summary).set_index('Model')
print("\n=== Metrics Summary ===")
print(metrics_df.round(4))

### 4.3 Cross-Validation

A single train/test split can be **lucky or unlucky** — the random split might happen to give us an unusually easy (or hard) test set.

**K-Fold Cross-Validation** gives a more reliable performance estimate by:

1. Splitting data into **K equal folds** (we use K=5)
2. Training K separate models, each time leaving one fold out as the test set
3. Reporting the **mean and standard deviation** across all K scores

```
Fold 1: [Test ] [Train] [Train] [Train] [Train]  → Score 1
Fold 2: [Train] [Test ] [Train] [Train] [Train]  → Score 2
Fold 3: [Train] [Train] [Test ] [Train] [Train]  → Score 3
Fold 4: [Train] [Train] [Train] [Test ] [Train]  → Score 4
Fold 5: [Train] [Train] [Train] [Train] [Test ]  → Score 5
                                                    ─────────
                                            Mean ± Std
```

In [ ]:
# 5-fold cross-validation for both models
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# For logistic regression, include scaling inside a Pipeline
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=500, random_state=42))
])

cv_models = {
    'Logistic Regression': (lr_pipeline, X_cls, y_cls),
    'Decision Tree (d=4)':  (DecisionTreeClassifier(max_depth=4, random_state=42), X_cls, y_cls)
}

cv_results = {}
print(f"{'Model':<28} {'Mean CV Acc':>12}  {'Std':>8}  {'95% CI':>18}")
print("-" * 72)

for name, (model, X, y) in cv_models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    cv_results[name] = scores
    ci_low  = scores.mean() - 1.96 * scores.std()
    ci_high = scores.mean() + 1.96 * scores.std()
    print(f"{name:<28} {scores.mean():>12.4f}  {scores.std():>8.4f}  "
          f"[{ci_low:.4f}, {ci_high:.4f}]")

In [ ]:
# Visualise cross-validation score distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
axes[0].boxplot(
    [cv_results[k] for k in cv_results],
    labels=list(cv_results.keys()),
    patch_artist=True,
    boxprops=dict(facecolor='#aed6f1', color='#1a5276'),
    medianprops=dict(color='#e74c3c', linewidth=2.5),
    whiskerprops=dict(color='#1a5276'),
    capprops=dict(color='#1a5276')
)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('5-Fold CV Score Distribution', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(0.5, 1.05)

# Fold-by-fold line plot
folds = range(1, 6)
for name, color in zip(cv_results, ['#3498db', '#e74c3c']):
    axes[1].plot(folds, cv_results[name], 'o-', linewidth=2, markersize=8,
                 color=color, label=name)
    axes[1].axhline(cv_results[name].mean(), linestyle='--', linewidth=1.5,
                    color=color, alpha=0.6)

axes[1].set_xlabel('Fold', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Accuracy per Fold', fontweight='bold')
axes[1].set_xticks(folds)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.5, 1.05)

fig.suptitle('Cross-Validation Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Part 5: Capstone Mini-Project

### African Development Pipeline: Predicting Access to Electricity

**Problem Statement:**  
Access to electricity is a key enabler of economic growth, education, and healthcare across Africa.  
In this capstone, we build a complete ML pipeline to **predict whether a country/region has high electricity access** (≥ 60% of population) from socioeconomic and infrastructure indicators.

**Pipeline Steps:**
1. Load & inspect data
2. Clean & engineer features
3. Visualise & explore
4. Preprocess (encode, scale, split)
5. Train multiple models
6. Evaluate & compare
7. Interpret results

> This mirrors the workflow you would follow in a real-world data science project.

In [ ]:
# ─── STEP 1: Load & Inspect ───────────────────────────────────────────────────
np.random.seed(99)
N = 300

regions_africa = [
    'North Africa', 'East Africa', 'West Africa',
    'Central Africa', 'Southern Africa'
]

# Electricity access differs significantly by region
region_elec = {
    'North Africa':   (85, 10),
    'Southern Africa': (65, 18),
    'East Africa':    (38, 18),
    'West Africa':    (40, 20),
    'Central Africa': (28, 15)
}

region_col      = np.random.choice(regions_africa, N, p=[0.15, 0.25, 0.28, 0.17, 0.15])
gdp_cap         = np.random.lognormal(mean=7.3, sigma=0.9, size=N)
rural_pct       = np.clip(np.random.normal(55, 18, N), 10, 90)
road_density    = np.clip(np.random.exponential(25, N), 1, 150)  # km per 1000 km²
mobile_subscr   = np.clip(np.random.normal(75, 25, N), 5, 150)   # per 100 people
gov_effectiveness = np.clip(np.random.normal(0, 0.8, N), -2.5, 2.5)  # World Bank scale
fdi_pct_gdp     = np.clip(np.random.exponential(3, N), 0, 20)    # FDI as % of GDP
renewable_pct   = np.clip(np.random.normal(30, 18, N), 0, 90)    # % of energy mix

# Electricity access: depends on region baseline + features
elec_access = np.array([
    np.clip(
        np.random.normal(region_elec[r][0], region_elec[r][1]) +
        0.003 * gdp_cap[i] +
        -0.15 * rural_pct[i] +
        0.10 * road_density[i] +
        0.08 * mobile_subscr[i] +
        5.0  * gov_effectiveness[i],
        2, 99
    )
    for i, r in enumerate(region_col)
])

# Binary target: High Electricity Access (>= 60%)
high_elec = (elec_access >= 60).astype(int)

# Introduce missing values (realistic data quality issue)
road_density_na = road_density.copy().astype(float)
road_density_na[np.random.choice(N, 20, replace=False)] = np.nan
fdi_na = fdi_pct_gdp.copy().astype(float)
fdi_na[np.random.choice(N, 15, replace=False)] = np.nan

capstone_df = pd.DataFrame({
    'region':            region_col,
    'gdp_per_capita':    np.round(gdp_cap, 0),
    'rural_pct':         np.round(rural_pct, 1),
    'road_density':      np.round(road_density_na, 1),
    'mobile_subscr':     np.round(mobile_subscr, 1),
    'gov_effectiveness': np.round(gov_effectiveness, 3),
    'fdi_pct_gdp':       np.round(fdi_na, 2),
    'renewable_pct':     np.round(renewable_pct, 1),
    'elec_access_pct':   np.round(elec_access, 1),
    'high_elec_access':  high_elec
})

print(f"Dataset: {capstone_df.shape[0]} rows x {capstone_df.shape[1]} columns")
print(f"\nClass distribution:")
print(capstone_df['high_elec_access'].value_counts()
      .rename({0: 'Low Access (<60%)', 1: 'High Access (>=60%)'}))
print(f"\nMissing values:")
print(capstone_df.isnull().sum())
capstone_df.head(10)

In [ ]:
# ─── STEP 2: Clean & Engineer Features ────────────────────────────────────────

df_clean = capstone_df.copy()

# 2a. Fill missing values with median (robust to outliers)
for col in ['road_density', 'fdi_pct_gdp']:
    median_val = df_clean[col].median()
    n_missing  = df_clean[col].isnull().sum()
    df_clean[col].fillna(median_val, inplace=True)
    print(f"Filled {n_missing} missing values in '{col}' with median = {median_val:.2f}")

# 2b. Feature engineering: log-transform skewed features
df_clean['log_gdp']   = np.log(df_clean['gdp_per_capita'])
df_clean['log_roads'] = np.log1p(df_clean['road_density'])  # log1p handles 0 values safely

# 2c. Encode region (one-hot)
df_encoded = pd.get_dummies(df_clean, columns=['region'], drop_first=True)

print(f"\nOriginal features: {capstone_df.shape[1]}")
print(f"After engineering: {df_encoded.shape[1]}")
print(f"\nMissing values after cleaning: {df_encoded.isnull().sum().sum()}")

In [ ]:
# ─── STEP 3: Explore & Visualise ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

palette2 = {0: '#e74c3c', 1: '#27ae60'}
label2   = {0: 'Low Access', 1: 'High Access'}

# Plot 1: Electricity access by region
region_order = capstone_df.groupby('region')['elec_access_pct'].median().sort_values().index
sns.boxplot(data=capstone_df, x='region', y='elec_access_pct',
            order=region_order, palette='viridis', ax=axes[0, 0])
axes[0, 0].axhline(60, color='red', linestyle='--', linewidth=2, label='60% threshold')
axes[0, 0].set_title('Electricity Access by Region', fontweight='bold')
axes[0, 0].set_xlabel('')
axes[0, 0].tick_params(axis='x', rotation=20)
axes[0, 0].legend(fontsize=9)

# Plot 2: GDP vs electricity access
for cls in [0, 1]:
    sub = capstone_df[capstone_df['high_elec_access'] == cls]
    axes[0, 1].scatter(sub['gdp_per_capita'], sub['elec_access_pct'],
                       color=palette2[cls], alpha=0.5, s=30, label=label2[cls])
axes[0, 1].axhline(60, color='black', linestyle='--', linewidth=1.5)
axes[0, 1].set_xlabel('GDP per Capita (USD)')
axes[0, 1].set_ylabel('Electricity Access (%)')
axes[0, 1].set_title('GDP vs Electricity Access', fontweight='bold')
axes[0, 1].legend(fontsize=9)

# Plot 3: Rural pct distribution by class
for cls in [0, 1]:
    sub = capstone_df[capstone_df['high_elec_access'] == cls]
    axes[0, 2].hist(sub['rural_pct'], bins=20, alpha=0.55,
                    color=palette2[cls], label=label2[cls], edgecolor='white')
axes[0, 2].set_xlabel('Rural Population (%)')
axes[0, 2].set_ylabel('Count')
axes[0, 2].set_title('Rural % by Electricity Access Class', fontweight='bold')
axes[0, 2].legend(fontsize=9)

# Plot 4: Gov. effectiveness vs electricity access
for cls in [0, 1]:
    sub = capstone_df[capstone_df['high_elec_access'] == cls]
    axes[1, 0].scatter(sub['gov_effectiveness'], sub['elec_access_pct'],
                       color=palette2[cls], alpha=0.5, s=30, label=label2[cls])
axes[1, 0].axhline(60, color='black', linestyle='--', linewidth=1.5)
axes[1, 0].set_xlabel('Government Effectiveness (World Bank)')
axes[1, 0].set_ylabel('Electricity Access (%)')
axes[1, 0].set_title('Governance vs Electricity Access', fontweight='bold')
axes[1, 0].legend(fontsize=9)

# Plot 5: Correlation heatmap (numeric features only)
numeric_cap = capstone_df.select_dtypes(include=[np.number]).drop(
    columns=['high_elec_access', 'elec_access_pct'])
corr_cap = numeric_cap.corrwith(capstone_df['high_elec_access']).sort_values()

colors_corr = ['#e74c3c' if c < 0 else '#27ae60' for c in corr_cap]
axes[1, 1].barh(corr_cap.index, corr_cap.values, color=colors_corr, edgecolor='white', height=0.6)
axes[1, 1].axvline(0, color='black', linewidth=0.8)
axes[1, 1].set_xlabel('Correlation with High Electricity Access')
axes[1, 1].set_title('Feature Correlations with Target', fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

# Plot 6: Mobile subscriptions by class
sns.violinplot(data=capstone_df, x='high_elec_access', y='mobile_subscr',
               palette=palette2, ax=axes[1, 2])
axes[1, 2].set_xticklabels(['Low Access', 'High Access'])
axes[1, 2].set_xlabel('')
axes[1, 2].set_ylabel('Mobile Subscriptions (per 100)')
axes[1, 2].set_title('Mobile Access by Electricity Class', fontweight='bold')

fig.suptitle('Capstone EDA: African Electricity Access',
             fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─── STEP 4: Preprocess ───────────────────────────────────────────────────────

# Define final feature set (exclude the raw electricity access — it would be data leakage!)
drop_cols = ['high_elec_access', 'elec_access_pct']
X_cap = df_encoded.drop(columns=drop_cols)
y_cap = df_encoded['high_elec_access']

# Train / test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cap, y_cap, test_size=0.20, random_state=42, stratify=y_cap
)

# Scale
sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr)
X_te_sc  = sc.transform(X_te)

print(f"Training   : {X_tr.shape[0]} samples")
print(f"Test       : {X_te.shape[0]} samples")
print(f"Features   : {X_cap.shape[1]}")
print(f"\nFeature columns:")
for col in X_cap.columns:
    print(f"  {col}")

In [ ]:
# ─── STEP 5: Train Three Models ───────────────────────────────────────────────

# Logistic Regression
lr_cap = LogisticRegression(max_iter=500, C=1.0, random_state=42)
lr_cap.fit(X_tr_sc, y_tr)
pred_lr_cap = lr_cap.predict(X_te_sc)

# Decision Tree
dt_cap = DecisionTreeClassifier(max_depth=5, min_samples_leaf=8, random_state=42)
dt_cap.fit(X_tr, y_tr)
pred_dt_cap = dt_cap.predict(X_te)

# Random Forest
rf_cap = RandomForestClassifier(
    n_estimators=150, max_depth=6, min_samples_leaf=5,
    random_state=42, n_jobs=-1
)
rf_cap.fit(X_tr, y_tr)
pred_rf_cap = rf_cap.predict(X_te)

# Cross-validation for all models
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cap_results = {}
print(f"{'Model':<25} {'Test Acc':>10}  {'CV Mean':>10}  {'CV Std':>8}")
print("-" * 58)

model_data = [
    ('Logistic Regression', lr_cap, X_te_sc,  X_tr_sc, pred_lr_cap),
    ('Decision Tree',       dt_cap, X_te,      X_tr,    pred_dt_cap),
    ('Random Forest',       rf_cap, X_te,      X_tr,    pred_rf_cap)
]

for name, model, X_test_m, X_train_m, preds in model_data:
    test_acc = accuracy_score(y_te, preds)
    cv_scores = cross_val_score(model, X_train_m, y_tr, cv=cv5, scoring='accuracy')
    cap_results[name] = {
        'predictions': preds,
        'test_acc':    test_acc,
        'cv_mean':     cv_scores.mean(),
        'cv_std':      cv_scores.std()
    }
    print(f"{name:<25} {test_acc:>10.4f}  {cv_scores.mean():>10.4f}  {cv_scores.std():>8.4f}")

In [ ]:
# ─── STEP 6: Evaluate & Compare ───────────────────────────────────────────────
fig = plt.figure(figsize=(20, 10))

# Row 1: Confusion matrices
model_labels = ['Logistic Regression', 'Decision Tree', 'Random Forest']
preds_list   = [pred_lr_cap, pred_dt_cap, pred_rf_cap]
class_lbl    = ['Low Access', 'High Access']

for i, (name, preds) in enumerate(zip(model_labels, preds_list)):
    ax = fig.add_subplot(2, 4, i + 1)
    cm = confusion_matrix(y_te, preds)
    ConfusionMatrixDisplay(cm, display_labels=class_lbl).plot(
        ax=ax, colorbar=False, cmap='Blues'
    )
    acc  = cap_results[name]['test_acc']
    f1   = f1_score(y_te, preds)
    ax.set_title(f'{name}\nAcc={acc:.3f}  F1={f1:.3f}', fontweight='bold', fontsize=10)

# Row 1, last column: Accuracy comparison bar chart
ax_bar = fig.add_subplot(2, 4, 4)
accs   = [cap_results[n]['test_acc'] for n in model_labels]
colors_bar = ['#3498db', '#e67e22', '#27ae60']
bars = ax_bar.bar(model_labels, accs, color=colors_bar, edgecolor='white', width=0.55)
for bar, acc in zip(bars, accs):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{acc:.3f}', ha='center', fontweight='bold', fontsize=11)
ax_bar.set_ylim(0, 1.12)
ax_bar.set_ylabel('Test Accuracy')
ax_bar.set_title('Model Comparison', fontweight='bold')
ax_bar.tick_params(axis='x', rotation=15)
ax_bar.grid(axis='y', alpha=0.3)

# Row 2: Random Forest feature importances (the best model)
ax_fi = fig.add_subplot(2, 4, (5, 8))
fi = pd.Series(rf_cap.feature_importances_, index=X_cap.columns).sort_values(ascending=True)
colors_fi = ['#27ae60' if v >= fi.quantile(0.6) else
              '#e67e22' if v >= fi.quantile(0.3) else
              '#95a5a6' for v in fi]
fi.plot(kind='barh', ax=ax_fi, color=colors_fi, edgecolor='white')
ax_fi.set_title('Random Forest — Feature Importance\n(green = most important)',
                fontweight='bold', fontsize=12)
ax_fi.set_xlabel('Importance Score')
ax_fi.grid(axis='x', alpha=0.3)

fig.suptitle('Capstone Results: Predicting High Electricity Access in Africa',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─── STEP 7: Interpret Results ────────────────────────────────────────────────

# Best model detailed report
best_name = max(cap_results, key=lambda k: cap_results[k]['test_acc'])
best_preds = cap_results[best_name]['predictions']

print(f"Best model: {best_name}")
print(f"Test Accuracy : {cap_results[best_name]['test_acc']:.4f}")
print(f"CV Accuracy   : {cap_results[best_name]['cv_mean']:.4f} ± {cap_results[best_name]['cv_std']:.4f}")
print()
print(classification_report(y_te, best_preds, target_names=['Low Access', 'High Access']))

# Top 5 most important features
top5 = pd.Series(rf_cap.feature_importances_, index=X_cap.columns).sort_values(ascending=False).head(5)
print("\nTop 5 predictors of high electricity access:")
for feat, imp in top5.items():
    print(f"  {feat:<30} {imp:.4f}")

In [ ]:
# Bonus: Use the model to score hypothetical new country profiles
new_countries = pd.DataFrame({
    'gdp_per_capita':         [12000, 800,  3500],
    'rural_pct':              [35,    75,   60],
    'road_density':           [80,    5,    30],
    'mobile_subscr':          [110,   40,   70],
    'gov_effectiveness':      [0.8,  -1.2,  0.0],
    'fdi_pct_gdp':            [5.0,   1.5,  3.0],
    'renewable_pct':          [25,    15,   45],
    'log_gdp':                [np.log(12000), np.log(800),  np.log(3500)],
    'log_roads':              [np.log1p(80),   np.log1p(5),  np.log1p(30)],
    # One-hot region columns (use 'North Africa' as reference — all zeros = North Africa)
    'region_East Africa':     [0, 1, 0],
    'region_North Africa':    [1, 0, 0],  # adjust if get_dummies dropped differently
    'region_Southern Africa': [0, 0, 0],
    'region_West Africa':     [0, 0, 1],
})

# Align to the same columns as training data
for col in X_cap.columns:
    if col not in new_countries.columns:
        new_countries[col] = 0
new_countries = new_countries[X_cap.columns]

# Predict
predictions_new = rf_cap.predict(new_countries)
proba_new       = rf_cap.predict_proba(new_countries)

print("Predictions for hypothetical country profiles:")
print(f"{'Profile':<10} {'P(Low)':>10} {'P(High)':>10} {'Prediction':>16}")
print("-" * 50)
profile_names = ['Country A\n(High GDP, Urban)',
                 'Country B\n(Low GDP, Rural)',
                 'Country C\n(Mid GDP, Mix)']
for i, (pred, proba) in enumerate(zip(predictions_new, proba_new)):
    outcome = 'High Access' if pred == 1 else 'Low Access'
    print(f"Profile {i+1}   {proba[0]:>10.4f} {proba[1]:>10.4f} {outcome:>16}")

### Capstone Reflection

In this full pipeline, we:

1. **Loaded** a 300-sample synthetic African development dataset with realistic missing values
2. **Cleaned** missing values using median imputation
3. **Engineered** features: log-transform of skewed variables, one-hot encoding of regions
4. **Explored** data with 6 plots linking socioeconomic indicators to electricity access
5. **Trained** three models: Logistic Regression, Decision Tree, Random Forest
6. **Evaluated** with confusion matrices, classification reports, and 5-fold cross-validation
7. **Interpreted** feature importances to understand what drives electricity access
8. **Scored** hypothetical new country profiles using the trained model

**Key Findings:**
- Government effectiveness and GDP per capita are strong predictors of electricity access
- Rural population percentage is negatively correlated with access
- Region matters — North Africa consistently outperforms Central Africa
- Random Forest typically yields the best accuracy due to its ensemble nature

---

---

## Part 6: Workshop Summary & Next Steps

### What We Covered Across 5 Days

| Day | Topic | Key Skills |
|-----|-------|------------|
| **Day 1** | Python Fundamentals | Variables, loops, functions, modules |
| **Day 2** | Data Structures & NumPy | Lists, dicts, arrays, vectorised operations |
| **Day 3** | Pandas & Data Wrangling | DataFrames, merging, groupby, cleaning |
| **Day 4** | Data Visualization | Matplotlib, Seaborn, EDA workflow |
| **Day 5** | Machine Learning | Regression, classification, evaluation, pipeline |

### Machine Learning — Quick Reference

| Concept | Summary |
|---------|--------|
| **Supervised Learning** | Learn from labeled (X, y) pairs |
| **Regression** | Predict a continuous number |
| **Classification** | Predict a category/class |
| **Train/Test Split** | Always evaluate on unseen data |
| **Overfitting** | Too complex → memorizes training data |
| **Cross-Validation** | K-fold for robust performance estimate |
| **Accuracy** | Simple but can mislead on imbalanced data |
| **Precision / Recall** | Use when false positives or false negatives have different costs |
| **F1-Score** | Balanced metric for imbalanced classes |
| **Feature Importance** | Which features drive predictions? |

### The scikit-learn Cheat Sheet

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LinearRegression, LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import accuracy_score, confusion_matrix, r2_score

# 1. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 2. Scale (for distance-based / gradient models)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# 3. Train
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# 4. Evaluate
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))
```

In [ ]:
# Final summary dashboard — a visual recap of what we built today
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Life Expectancy model (regression)
axes[0].scatter(y_test_l, y_pred_l, color='#1a6e9e', alpha=0.8, s=60,
               edgecolors='white', linewidth=0.8, zorder=5)
lims_l = [min(y_test_l.min(), y_pred_l.min()) - 1,
           max(y_test_l.max(), y_pred_l.max()) + 1]
axes[0].plot(lims_l, lims_l, 'r--', linewidth=2)
axes[0].set_xlabel('Actual Life Expectancy (years)')
axes[0].set_ylabel('Predicted (years)')
axes[0].set_title(f'Part 2: Linear Regression\nR² = {r2:.3f}  RMSE = {rmse:.2f} yrs',
                  fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Panel 2: Classification comparison (Part 3 & 4)
model_names_bar = ['Logistic\nRegression', 'Decision\nTree (d=4)']
accs_bar        = [acc_lr, acc_dt]
colors_b        = ['#3498db', '#e67e22']
bars2 = axes[1].bar(model_names_bar, accs_bar, color=colors_b, edgecolor='white', width=0.45)
for bar, acc in zip(bars2, accs_bar):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{acc:.3f}', ha='center', fontweight='bold', fontsize=13)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Part 3: Classification\nDevelopment Level Prediction',
                  fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

# Panel 3: Capstone model comparison
cap_names  = ['Logistic\nReg.', 'Decision\nTree', 'Random\nForest']
cap_accs   = [cap_results[k]['test_acc'] for k in cap_results]
colors_cap = ['#3498db', '#e67e22', '#27ae60']
bars3 = axes[2].bar(cap_names, cap_accs, color=colors_cap, edgecolor='white', width=0.45)
for bar, acc in zip(bars3, cap_accs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{acc:.3f}', ha='center', fontweight='bold', fontsize=13)
axes[2].set_ylim(0, 1.15)
axes[2].set_ylabel('Test Accuracy')
axes[2].set_title('Part 5: Capstone\nElectricity Access Prediction',
                  fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)

fig.suptitle('Day 5 Summary — Models Built Today', fontsize=16, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

print("\nCongratulations on completing the 5-day Python for Data Science Workshop!")

---

## Recommended Next Steps

### Immediate Practice (This Week)

1. **Kaggle** — Join beginner competitions at [kaggle.com](https://www.kaggle.com). Start with *Titanic* (classification) or *House Prices* (regression).
2. **Try a new dataset** — Download a dataset from your country's national statistics bureau and repeat the capstone pipeline.
3. **Extend today's notebooks** — Add a `GradientBoostingClassifier` to the capstone and compare results.

### Topics to Explore Next

| Topic | Why It Matters |
|-------|----------------|
| **Hyperparameter Tuning** (`GridSearchCV`) | Find the best settings for your model |
| **Pipelines** (`sklearn.pipeline`) | Chain preprocessing + model cleanly |
| **Gradient Boosting** (XGBoost, LightGBM) | State-of-the-art for tabular data |
| **Feature Selection** | Remove irrelevant features, improve performance |
| **Imbalanced Classes** (SMOTE, class_weight) | Handle unequal class distributions |
| **Neural Networks** (PyTorch, TensorFlow) | Deep learning for images, text, audio |
| **NLP** (spaCy, HuggingFace) | Analyse text data in local languages |
| **Time Series** (statsmodels, Prophet) | Forecast economic or climate data |

### Free Learning Resources

- **scikit-learn docs**: [scikit-learn.org](https://scikit-learn.org) — excellent tutorials and API reference
- **Kaggle Learn**: [kaggle.com/learn](https://www.kaggle.com/learn) — free micro-courses on ML, SQL, deep learning
- **fast.ai**: [fast.ai](https://www.fast.ai) — practical deep learning with Python
- **African Data**: [dataportal.opendataforafrica.org](https://dataportal.opendataforafrica.org) — real African datasets
- **World Bank Open Data**: [data.worldbank.org](https://data.worldbank.org) — development indicators for all countries

---

### Exercises — Try These on Your Own!

1. **Regression exercise:** Add `urban_pct` as a feature to the life expectancy model. Does R² improve?
2. **Hyperparameter tuning:** Use `GridSearchCV` to find the optimal `max_depth` and `n_estimators` for the Random Forest capstone model.
3. **ROC Curve:** Plot Receiver Operating Characteristic curves for all three capstone models on the same axes (hint: `sklearn.metrics.roc_curve`).
4. **New target:** Change the capstone target to predict whether a country has **mobile internet penetration > 50%**. Rebuild the entire pipeline.
5. **Real data:** Download the World Bank's *Africa Development Indicators* dataset and apply what you learned today.

---

*Python for Data Science Workshop | Day 5 of 5*  
*Dr. Yaé Ulrich Gaba — AIRINA Labs / ACAS*

---

In [ ]:
# Your workspace — use this cell for exercises!
# Exercise 1: Hyperparameter tuning

from sklearn.model_selection import GridSearchCV

# Uncomment and complete:
# param_grid = {
#     'n_estimators': [50, 100, 200],
#     'max_depth':    [3, 5, 8, None]
# }
# grid_search = GridSearchCV(RandomForestClassifier(random_state=42),
#                            param_grid, cv=5, scoring='accuracy', n_jobs=-1)
# grid_search.fit(X_tr, y_tr)
# print("Best params:", grid_search.best_params_)
# print("Best CV score:", grid_search.best_score_)

print("Fill in the grid search above and run!")

In [ ]:
# Exercise 2: ROC Curve
from sklearn.metrics import roc_curve, auc

fig, ax = plt.subplots(figsize=(8, 6))

roc_models = [
    ('Logistic Regression', lr_cap,  X_te_sc),
    ('Decision Tree',       dt_cap,  X_te),
    ('Random Forest',       rf_cap,  X_te),
]

colors_roc = ['#3498db', '#e67e22', '#27ae60']

for (name, model, X_test_m), color in zip(roc_models, colors_roc):
    proba_pos = model.predict_proba(X_test_m)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, proba_pos)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2.5, color=color,
            label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random classifier (AUC=0.5)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves — Electricity Access Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAUC Interpretation:")
print("  1.0  = Perfect model")
print("  0.5  = No better than random guessing")
print("  >0.8 = Good model")